# Experiment 10b: Additive-Only Q-Matrix Refinement

**Rationale:** Experiment 10a showed that unconstrained LLM refinement over-prunes KC tags
(71 removals, 5 additions), reducing PFA AUC from 0.774 to 0.766. The LLM removed KCs like
NestedIf (gamma = 0.24) that PFA was actively using as learning signals.

**Fix:** Keep ALL instructor tags and only let the LLM ADD KCs it sees evidence for in failing
student code. This preserves all existing PFA signal while potentially introducing new useful tags.

**Results saved to:** `results/10_qmatrix_refinement/`

In [1]:
import pandas as pd
import numpy as np
import json
import time
import os
from pathlib import Path
from collections import defaultdict
from dotenv import load_dotenv
from google import genai
from google.genai import types
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

# === CONFIGURATION ===
PROJECT_ROOT = Path("/mnt/d/Projects/kintsugi")
DATA_DIR = PROJECT_ROOT / "dataset" / "CodeWorkout"
RESULTS_DIR = PROJECT_ROOT / "results" / "10_qmatrix_refinement"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
MODEL_ID = "gemini-2.0-flash"
SLEEP_SECONDS = 1.5
RANDOM_SEED = 42
N_FOLDS = 5

np.random.seed(RANDOM_SEED)

if not GEMINI_API_KEY:
    print("WARNING: GEMINI_API_KEY not found in environment.")
    client = None
else:
    client = genai.Client(api_key=GEMINI_API_KEY)
    print(f"Gemini client initialized with model {MODEL_ID}")

Gemini client initialized with model gemini-2.0-flash


## Step 1: Load data

In [2]:
print("Loading data...")

MAINTABLE_PATH = DATA_DIR / "MainTable.csv"
SUBJECT_TABLE_PATH = DATA_DIR / "LinkTables" / "Subject.csv"
CODESTATES_TABLE_PATH = DATA_DIR / "LinkTables" / "CodeStates.csv"
PROBLEM_PROMPT_PATH = DATA_DIR / "Problem_Prompts" / "problem_prompts.csv"

main_df = pd.read_csv(MAINTABLE_PATH)
subject_df = pd.read_csv(SUBJECT_TABLE_PATH)
codestates_df = pd.read_csv(CODESTATES_TABLE_PATH)
pp_df = pd.read_csv(PROBLEM_PROMPT_PATH)

# Merge to get subject info and code
df = main_df[main_df["EventType"] == "Run.Program"].merge(
    subject_df, on="SubjectID"
).merge(
    codestates_df, on="CodeStateID"
)

# Best attempt per student per problem
best = df.sort_values("Score", ascending=False).drop_duplicates(
    subset=["SubjectID", "ProblemID"], keep="first"
)
best["passed"] = (best["Score"] == 1.0).astype(int)

# KC columns
KC_COLS = list(pp_df.columns[3:])
ALL_KCS = KC_COLS

print(f"Dataset: {len(best)} best attempts across {best['SubjectID'].nunique()} students")
print(f"Problems in Q-matrix: {pp_df['ProblemID'].nunique()}")
print(f"Knowledge Components ({len(ALL_KCS)}): {ALL_KCS}")

Loading data...
Dataset: 15375 best attempts across 372 students
Problems in Q-matrix: 50
Knowledge Components (18): ['If/Else', 'NestedIf', 'While', 'For', 'NestedFor', 'Math+-*/', 'Math%', 'LogicAndNotOr', 'LogicCompareNum', 'LogicBoolean', 'StringFormat', 'StringConcat', 'StringIndex', 'StringLen', 'StringEqual', 'CharEqual', 'ArrayIndex', 'DefFunction']


## Step 2: Sampling helpers

In [3]:
MIN_CODE_LENGTH = 50
MIN_CODE_LINES = 3

def is_genuine_attempt(code_str):
    if pd.isna(code_str):
        return False
    code = str(code_str).strip()
    if len(code) < MIN_CODE_LENGTH:
        return False
    if code.count('\n') + 1 < MIN_CODE_LINES:
        return False
    return True

def get_problem_difficulty(problem_id, best_df):
    prob_data = best_df[best_df["ProblemID"] == problem_id]
    total = len(prob_data)
    if total == 0:
        return {"pass_rate": 0, "avg_score": 0, "total_attempts": 0, "difficulty": "unknown"}
    pass_rate = prob_data["passed"].mean()
    avg_score = prob_data["Score"].mean()
    if pass_rate >= 0.8:
        difficulty = "easy"
    elif pass_rate >= 0.5:
        difficulty = "medium"
    else:
        difficulty = "hard"
    return {
        "pass_rate": round(pass_rate, 3),
        "avg_score": round(avg_score, 3),
        "total_attempts": total,
        "difficulty": difficulty,
    }

def get_code_samples(problem_id, best_df, n_pass=5, n_fail=5):
    prob_data = best_df[best_df["ProblemID"] == problem_id].copy()
    prob_data = prob_data[prob_data["Code"].apply(is_genuine_attempt)]

    passing = prob_data[prob_data["passed"] == 1]
    failing = prob_data[prob_data["passed"] == 0]

    partial_fails = failing[failing["Score"] > 0].sort_values("Score", ascending=False)
    zero_fails = failing[failing["Score"] == 0]

    if len(partial_fails) >= n_fail:
        fail_sample = partial_fails.sample(n=n_fail, random_state=RANDOM_SEED)
    else:
        remaining = n_fail - len(partial_fails)
        zero_sample = zero_fails.sample(n=min(remaining, len(zero_fails)), random_state=RANDOM_SEED) if len(zero_fails) > 0 else zero_fails
        fail_sample = pd.concat([partial_fails, zero_sample])

    if len(passing) >= n_pass:
        pass_sample = passing.sample(n=n_pass, random_state=RANDOM_SEED)
    else:
        pass_sample = passing

    return pass_sample, fail_sample

print("Sampling helpers ready.")

Sampling helpers ready.


## Step 3: Additive-only prompt

Key difference from 10a: the LLM can **only ADD** KCs. All instructor tags are preserved.

In [4]:
def build_qmatrix_prompt_additive(problem_id, requirement, instructor_kcs,
                                   passing_codes, passing_scores,
                                   failing_codes, failing_scores,
                                   difficulty_info):
    """Build the ADDITIVE-ONLY prompt for Q-matrix refinement.
    
    Key difference from 10a: the LLM can ONLY ADD KCs, never remove.
    All instructor tags are preserved.
    """

    pass_section = ""
    for i, (code, score) in enumerate(zip(passing_codes, passing_scores), 1):
        pass_section += f"\n--- Passing Student {i} (Score: {score:.2f} = all test cases correct) ---\n{code}\n"

    fail_section = ""
    for i, (code, score) in enumerate(zip(failing_codes, failing_scores), 1):
        pct = int(score * 100)
        fail_label = f"{pct}% of test cases passed" if score > 0 else "0% — failed all test cases"
        fail_section += f"\n--- Failing Student {i} (Score: {score:.2f} = {fail_label}) ---\n{code}\n"

    untagged_kcs = [kc for kc in ALL_KCS if kc not in instructor_kcs]

    prompt = f"""You are an expert CS education researcher analyzing a Java programming problem to determine if any Knowledge Components (KCs) are MISSING from the instructor's tags.

PROBLEM (ID: {problem_id}):
{requirement}

PROBLEM DIFFICULTY:
- Class pass rate: {difficulty_info['pass_rate']*100:.1f}% of students scored 100%
- Class average score: {difficulty_info['avg_score']*100:.1f}%
- Total students who attempted: {difficulty_info['total_attempts']}
- Difficulty level: {difficulty_info['difficulty']}

INSTRUCTOR'S CURRENT KC TAGS FOR THIS PROBLEM:
{json.dumps(instructor_kcs)}

KCs NOT CURRENTLY TAGGED (candidates for addition):
{json.dumps(untagged_kcs)}

THE FULL LIST OF 18 POSSIBLE KCs:
{json.dumps(ALL_KCS)}

Below are real student code submissions. All submissions shown are GENUINE ATTEMPTS where students wrote meaningful code (not just a return statement). Each student's score indicates what percentage of automated test cases they passed. A score below 1.0 means at least one test case failed.

=== PASSING STUDENT CODE (scored 100% — all test cases correct) ===
{pass_section}

=== FAILING STUDENT CODE (scored below 100%) ===
{fail_section}

YOUR TASK:
The instructor's existing KC tags are KEPT AS-IS. Do NOT suggest removing any of them.

Your job is to identify MISSING KCs — skills that the instructor did not tag but that failing students clearly struggle with. Look at the specific errors in failing code and determine if any of the untagged KCs are actually being tested by this problem.

GUIDELINES FOR ADDING A KC:
1. Only add a KC if you see clear evidence in the FAILING student code that students struggle with it. "Clear evidence" means at least 2 out of the failing students show errors related to that KC.
2. Do NOT add a KC just because it appears in the code. It must be a source of FAILURE — something failing students get wrong that passing students get right.
3. Pay attention to partial scores. A student who scores 70% is almost correct — the missing KC is very specific. Look for what exactly they got wrong.
4. Common candidates to check:
   - LogicBoolean: Are failing students confused about boolean logic (using = instead of ==, not understanding true/false)?
   - Math%: Are failing students missing modulo operations?
   - DefFunction: Does the problem require a helper method that failing students don't define?
   - CharEqual: Are failing students comparing characters incorrectly?
5. It is COMPLETELY FINE to add zero KCs if the instructor's tags already cover everything. Do not force additions.

RESPOND WITH ONLY a JSON object in this exact format, no other text:
{{
    "problem_id": {problem_id},
    "added_kcs": ["KC1", "KC2"],
    "reasoning": "Brief explanation of what evidence you found in failing code for each added KC. If no additions, explain why the instructor's tags are sufficient."
}}"""

    return prompt

print("Additive prompt function defined.")

Additive prompt function defined.


## Step 4: Run the additive refinement

In [5]:
results = []
errors = []

problem_ids = sorted(pp_df["ProblemID"].unique())
total = len(problem_ids)

print(f"Starting ADDITIVE Q-matrix refinement: {total} problems")
print("=" * 60)

for idx, pid in enumerate(problem_ids):
    row = pp_df[pp_df["ProblemID"] == pid].iloc[0]
    requirement = row["Requirement"]
    instructor_kcs = [kc for kc in KC_COLS if row[kc] == 1]

    pass_sample, fail_sample = get_code_samples(pid, best)
    passing_codes = pass_sample["Code"].tolist() if len(pass_sample) > 0 else []
    passing_scores = pass_sample["Score"].tolist() if len(pass_sample) > 0 else []
    failing_codes = fail_sample["Code"].tolist() if len(fail_sample) > 0 else []
    failing_scores = fail_sample["Score"].tolist() if len(fail_sample) > 0 else []

    difficulty_info = get_problem_difficulty(pid, best)

    # Skip if no genuine failing attempts
    if len(failing_codes) == 0:
        print(f"  [{idx+1}/{total}] Problem {pid}: SKIPPED (no failures)")
        results.append({
            "ProblemID": pid,
            "Instructor_KCs": json.dumps(instructor_kcs),
            "Final_KCs": json.dumps(instructor_kcs),
            "Added_KCs": json.dumps([]),
            "Reasoning": "No genuine failing submissions",
            "Num_Instructor": len(instructor_kcs),
            "Num_Final": len(instructor_kcs),
            "Num_Added": 0,
            "TimeSec": 0,
            "Raw_Response": "SKIPPED",
        })
        continue

    # Build additive prompt
    prompt = build_qmatrix_prompt_additive(
        pid, requirement, instructor_kcs,
        passing_codes, passing_scores,
        failing_codes, failing_scores,
        difficulty_info
    )

    start_time = time.time()
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.2,
                max_output_tokens=1024,
            )
        )

        duration = time.time() - start_time
        raw_text = response.text.strip()
        clean = raw_text.replace("```json", "").replace("```", "").strip()
        parsed = json.loads(clean)

        added = parsed.get("added_kcs", [])
        reasoning = parsed.get("reasoning", "")

        # Final KCs = instructor + additions (no removals)
        final_kcs = list(set(instructor_kcs + added))

        results.append({
            "ProblemID": pid,
            "Instructor_KCs": json.dumps(instructor_kcs),
            "Final_KCs": json.dumps(final_kcs),
            "Added_KCs": json.dumps(added),
            "Reasoning": reasoning,
            "Num_Instructor": len(instructor_kcs),
            "Num_Final": len(final_kcs),
            "Num_Added": len(added),
            "TimeSec": round(duration, 2),
            "Raw_Response": raw_text,
        })

        add_str = f"+{len(added)} KCs: {added}" if added else "no changes"
        status = f"OK: {len(instructor_kcs)} -> {len(final_kcs)} ({add_str})"

    except Exception as e:
        duration = time.time() - start_time
        errors.append({"ProblemID": pid, "Error": str(e)})
        status = f"ERROR: {str(e)[:50]}"

    eta = (duration * (total - idx - 1)) / 60
    print(f"  [{idx+1}/{total}] Problem {pid}: {status} ({duration:.1f}s, ETA: {eta:.1f}min)")

    time.sleep(SLEEP_SECONDS)

print(f"\nDone! {len(results)} successful, {len(errors)} errors")

results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_DIR / "qmatrix_additive_results.csv", index=False)
print(f"Saved to {RESULTS_DIR / 'qmatrix_additive_results.csv'}")

Starting ADDITIVE Q-matrix refinement: 50 problems
  [1/50] Problem 1: OK: 4 -> 4 (+1 KCs: ['If/Else']) (1.9s, ETA: 1.5min)
  [2/50] Problem 3: OK: 5 -> 5 (no changes) (1.9s, ETA: 1.5min)
  [3/50] Problem 5: OK: 3 -> 5 (+2 KCs: ['LogicAndNotOr', 'LogicCompareNum']) (1.9s, ETA: 1.5min)
  [4/50] Problem 12: OK: 5 -> 5 (no changes) (1.5s, ETA: 1.1min)
  [5/50] Problem 13: OK: 6 -> 6 (no changes) (1.3s, ETA: 0.9min)
  [6/50] Problem 17: OK: 4 -> 4 (no changes) (1.5s, ETA: 1.1min)
  [7/50] Problem 20: OK: 5 -> 5 (no changes) (1.4s, ETA: 1.0min)
  [8/50] Problem 21: OK: 5 -> 5 (+1 KCs: ['If/Else']) (2.5s, ETA: 1.7min)
  [9/50] Problem 22: OK: 6 -> 6 (no changes) (1.2s, ETA: 0.8min)
  [10/50] Problem 24: OK: 4 -> 5 (+1 KCs: ['Math%']) (1.3s, ETA: 0.9min)
  [11/50] Problem 25: OK: 4 -> 5 (+1 KCs: ['LogicBoolean']) (1.4s, ETA: 0.9min)
  [12/50] Problem 28: OK: 6 -> 7 (+1 KCs: ['DefFunction']) (2.2s, ETA: 1.4min)
  [13/50] Problem 31: ERROR: Invalid control character at: line 4 column 362 (c (1.

## Step 5: Build the additive Q-matrix

In [6]:
additive_qmatrix = []

for _, row in results_df.iterrows():
    pid = row["ProblemID"]
    final_kcs = json.loads(row["Final_KCs"])

    qrow = {"ProblemID": pid}
    for kc in ALL_KCS:
        qrow[kc] = 1 if kc in final_kcs else 0
    additive_qmatrix.append(qrow)

additive_df = pd.DataFrame(additive_qmatrix)
additive_df.to_csv(RESULTS_DIR / "additive_problem_prompts.csv", index=False)

print("=== Additive Q-Matrix Summary ===")
print(f"Problems: {len(additive_df)}")
total_added = 0
for kc in ALL_KCS:
    orig = int(pp_df[kc].sum()) if kc in pp_df.columns else 0
    new = int(additive_df[kc].sum())
    delta = new - orig
    total_added += max(delta, 0)
    marker = f"  (+{delta})" if delta > 0 else ""
    print(f"  {kc:20s}: {orig:2d} -> {new:2d}{marker}")

print(f"\nTotal tags added:   {total_added}")
print(f"Total tags removed: 0 (by design)")
print(f"Average KCs per problem: {pp_df[KC_COLS].sum(axis=1).mean():.1f} -> {additive_df[ALL_KCS].sum(axis=1).mean():.1f}")

print(f"\n=== Problems with additions ===")
had_additions = False
for _, row in results_df.iterrows():
    added = json.loads(row["Added_KCs"])
    if added:
        had_additions = True
        print(f"  Problem {row['ProblemID']}: +{added}")
        print(f"    Reason: {row['Reasoning'][:120]}...")
if not had_additions:
    print("  (none — instructor tags deemed sufficient for all problems)")

=== Additive Q-Matrix Summary ===
Problems: 49
  If/Else             : 44 -> 46  (+2)
  NestedIf            : 10 -> 11  (+1)
  While               :  3 ->  6  (+3)
  For                 : 27 -> 26
  NestedFor           :  4 ->  4
  Math+-*/            : 20 -> 20
  Math%               :  5 ->  7  (+2)
  LogicAndNotOr       : 30 -> 33  (+3)
  LogicCompareNum     : 39 -> 40  (+1)
  LogicBoolean        :  6 -> 17  (+11)
  StringFormat        : 14 -> 13
  StringConcat        :  5 ->  5
  StringIndex         : 12 -> 11
  StringLen           :  9 ->  8
  StringEqual         :  7 ->  8  (+1)
  CharEqual           :  4 ->  6  (+2)
  ArrayIndex          : 20 -> 20
  DefFunction         :  2 ->  4  (+2)

Total tags added:   28
Total tags removed: 0 (by design)
Average KCs per problem: 5.2 -> 5.8

=== Problems with additions ===
  Problem 1: +['If/Else']
    Reason: Failing Student 1 completely omits the if/else logic, returning the sum regardless of the condition. Failing Student 2 h...
  Problem

## Step 6: PFA evaluation

Plug the additive Q-matrix into PFA and compare against instructor baseline.

In [7]:
# Load first attempts for PFA (chronological order, one per student per problem)
mt = pd.read_csv(DATA_DIR / "MainTable.csv")
cs = pd.read_csv(CODESTATES_TABLE_PATH)

mt = mt[mt['EventType'] == 'Run.Program'].copy()
mt = mt.dropna(subset=['Score'])
mt['ServerTimestamp'] = pd.to_datetime(mt['ServerTimestamp'])
mt.sort_values(['SubjectID', 'ServerTimestamp'], inplace=True)
mt['correct'] = (mt['Score'] == 1.0).astype(int)
first_attempts = mt.drop_duplicates(subset=['SubjectID', 'ProblemID'], keep='first').copy()
first_attempts = first_attempts.reset_index(drop=True)

print(f"First attempts: {len(first_attempts)} rows, "
      f"{first_attempts['SubjectID'].nunique()} students, "
      f"{first_attempts['ProblemID'].nunique()} problems")
print(f"Correct rate: {first_attempts['correct'].mean():.3f}")

First attempts: 16179 rows, 413 students, 50 problems
Correct rate: 0.403


In [8]:
# Build KC maps

# Instructor KC map (baseline)
instructor_kc_map = {}
for _, row in pp_df.iterrows():
    pid = row['ProblemID']
    kcs = [kc for kc in KC_COLS if row[kc] == 1]
    instructor_kc_map[pid] = kcs
instructor_all_kcs = sorted(KC_COLS)

# Additive KC map
additive_pp = pd.read_csv(RESULTS_DIR / "additive_problem_prompts.csv")
additive_kc_map = {}
for _, row in additive_pp.iterrows():
    pid = row['ProblemID']
    kcs = [kc for kc in ALL_KCS if row[kc] == 1]
    additive_kc_map[pid] = kcs
additive_all_kcs = sorted(set(kc for kcs in additive_kc_map.values() for kc in kcs))

print(f"Instructor KCs: {len(instructor_all_kcs)} unique, {len(instructor_kc_map)} problems")
print(f"Additive KCs:   {len(additive_all_kcs)} unique, {len(additive_kc_map)} problems")

Instructor KCs: 18 unique, 50 problems
Additive KCs:   18 unique, 49 problems


In [9]:
def build_pfa_features(submissions_df, kc_map, all_kcs):
    """Build PFA features: prior success/failure counts per KC for each submission."""
    kc_to_idx = {kc: i for i, kc in enumerate(all_kcs)}
    n_kcs = len(all_kcs)

    X_rows, y_rows, student_ids = [], [], []
    skipped = 0

    for student_id, student_df in submissions_df.groupby('SubjectID'):
        success_counts = np.zeros(n_kcs)
        failure_counts = np.zeros(n_kcs)

        for _, row in student_df.iterrows():
            pid = row['ProblemID']
            correct = row['correct']

            if pid not in kc_map or len(kc_map[pid]) == 0:
                skipped += 1
                continue

            problem_kcs = kc_map[pid]
            feature_vec = np.zeros(2 * n_kcs)
            for kc in problem_kcs:
                if kc in kc_to_idx:
                    idx = kc_to_idx[kc]
                    feature_vec[2 * idx] = success_counts[idx]
                    feature_vec[2 * idx + 1] = failure_counts[idx]

            X_rows.append(feature_vec)
            y_rows.append(correct)
            student_ids.append(student_id)

            for kc in problem_kcs:
                if kc in kc_to_idx:
                    idx = kc_to_idx[kc]
                    if correct == 1:
                        success_counts[idx] += 1
                    else:
                        failure_counts[idx] += 1

    if skipped > 0:
        print(f"  Skipped {skipped} submissions (no KC mapping)")

    return np.array(X_rows), np.array(y_rows), student_ids


def run_pfa_cv(X, y, student_ids, n_folds=5, seed=42):
    """Run PFA (logistic regression) with student-level stratified K-fold CV."""
    unique_students = list(set(student_ids))
    sid_array = np.array(student_ids)

    student_avg = {s: y[sid_array == s].mean() for s in unique_students}
    student_labels = np.array([1 if student_avg[s] >= 0.5 else 0 for s in unique_students])

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    aucs, f1s, accs = [], [], []

    for fold, (train_idx, test_idx) in enumerate(skf.split(unique_students, student_labels), 1):
        train_students = set(np.array(unique_students)[train_idx])
        test_students = set(np.array(unique_students)[test_idx])

        train_mask = np.array([s in train_students for s in student_ids])
        test_mask = np.array([s in test_students for s in student_ids])

        model = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=seed)
        model.fit(X[train_mask], y[train_mask])

        y_prob = model.predict_proba(X[test_mask])[:, 1]
        y_pred = model.predict(X[test_mask])

        aucs.append(roc_auc_score(y[test_mask], y_prob))
        f1s.append(f1_score(y[test_mask], y_pred))
        accs.append(accuracy_score(y[test_mask], y_pred))
        print(f"  Fold {fold}: AUC={aucs[-1]:.4f}, F1={f1s[-1]:.4f}, Acc={accs[-1]:.4f}")

    return {
        'AUC': (np.mean(aucs), np.std(aucs)),
        'F1': (np.mean(f1s), np.std(f1s)),
        'Accuracy': (np.mean(accs), np.std(accs)),
        'per_fold_auc': aucs,
    }


print("Building PFA features...")

print(f"\nInstructor ({len(instructor_all_kcs)} KCs):")
X_inst, y_inst, sids_inst = build_pfa_features(first_attempts, instructor_kc_map, instructor_all_kcs)
print(f"  X shape: {X_inst.shape}")

print(f"\nAdditive ({len(additive_all_kcs)} KCs):")
X_add, y_add, sids_add = build_pfa_features(first_attempts, additive_kc_map, additive_all_kcs)
print(f"  X shape: {X_add.shape}")

Building PFA features...

Instructor (18 KCs):
  X shape: (16179, 36)

Additive (18 KCs):
  Skipped 339 submissions (no KC mapping)
  X shape: (15840, 36)


In [10]:
print("=" * 70)
print("Running PFA Cross-Validation")
print("=" * 70)

print(f"\n--- Instructor KCs ({len(instructor_all_kcs)} KCs) ---")
results_inst = run_pfa_cv(X_inst, y_inst, sids_inst, N_FOLDS, RANDOM_SEED)

print(f"\n--- Additive KCs ({len(additive_all_kcs)} KCs) ---")
results_add = run_pfa_cv(X_add, y_add, sids_add, N_FOLDS, RANDOM_SEED)

Running PFA Cross-Validation

--- Instructor KCs (18 KCs) ---
  Fold 1: AUC=0.7805, F1=0.5923, Acc=0.7224
  Fold 2: AUC=0.7795, F1=0.5889, Acc=0.7344
  Fold 3: AUC=0.7744, F1=0.5862, Acc=0.7192
  Fold 4: AUC=0.7623, F1=0.5818, Acc=0.7097
  Fold 5: AUC=0.7720, F1=0.5903, Acc=0.7217

--- Additive KCs (18 KCs) ---
  Fold 1: AUC=0.7713, F1=0.5775, Acc=0.7184
  Fold 2: AUC=0.7699, F1=0.5738, Acc=0.7112
  Fold 3: AUC=0.7672, F1=0.5757, Acc=0.7163
  Fold 4: AUC=0.7905, F1=0.5996, Acc=0.7417
  Fold 5: AUC=0.7529, F1=0.5710, Acc=0.7145


In [11]:
print("\n" + "=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)

print(f"\n{'KC Model':<35} {'AUC':>16} {'F1':>16} {'Accuracy':>16}")
print("-" * 85)

all_results = [
    (f"Instructor ({len(instructor_all_kcs)} KCs)", results_inst),
    (f"Additive ({len(additive_all_kcs)} KCs)", results_add),
]

for name, res in all_results:
    auc_m, auc_s = res['AUC']
    f1_m, f1_s = res['F1']
    acc_m, acc_s = res['Accuracy']
    print(f"{name:<35} {auc_m:.4f}±{auc_s:.4f} {f1_m:.4f}±{f1_s:.4f} {acc_m:.4f}±{acc_s:.4f}")

# Delta
delta_auc = results_add['AUC'][0] - results_inst['AUC'][0]
direction = "+" if delta_auc >= 0 else ""
print(f"\nAdditive vs Instructor ΔAUC = {direction}{delta_auc:.4f}")

# Paired t-test
t_stat, p_val = stats.ttest_rel(results_add['per_fold_auc'], results_inst['per_fold_auc'])
sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
print(f"Paired t-test: t={t_stat:.3f}, p={p_val:.4f} [{sig}]")

# Summary stats on additions
n_with_additions = sum(1 for _, r in results_df.iterrows() if json.loads(r['Added_KCs']))
total_added_tags = results_df['Num_Added'].sum()
print(f"\nQ-matrix changes:")
print(f"  Problems with additions: {n_with_additions}/{len(results_df)}")
print(f"  Total tags added:        {total_added_tags}")
print(f"  Total tags removed:      0 (by design)")


RESULTS SUMMARY

KC Model                                         AUC               F1         Accuracy
-------------------------------------------------------------------------------------
Instructor (18 KCs)                 0.7737±0.0065 0.5879±0.0037 0.7215±0.0079
Additive (18 KCs)                   0.7703±0.0120 0.5795±0.0103 0.7204±0.0109

Additive vs Instructor ΔAUC = -0.0034
Paired t-test: t=-0.417, p=0.6981 [ns]

Q-matrix changes:
  Problems with additions: 32/49
  Total tags added:        36
  Total tags removed:      0 (by design)
